In [1]:
"""
Text Duplication Features Extraction (Coordination Detection)
==============================================================
检测文本重复/相似，识别复制粘贴协同行为

输入: reddit_wsb_for_network.csv
输出: text_duplication_features_5min.parquet

核心特征（6列）:
1. exact_duplicate_rate - 完全相同文本比例
2. fuzzy_duplicate_rate - 高度相似文本比例（similarity>0.8）
3. unique_text_ratio - 独特文本比例
4. avg_text_similarity - 平均文本相似度
5. max_duplicate_count - 单个文本最大重复次数
6. duplicate_cluster_size - 重复文本簇大小

理论依据:
- 操纵者: 复制粘贴相同口号/链接
- 有机讨论: 原创性高，文本多样
- 检测copy-paste coordination

注意: 
- 虽然诊断分数低(5.7)，但可能在特定窗口有强信号
- 使用简化版实现，计算成本低
"""

import pandas as pd
import numpy as np
from datetime import datetime
from collections import Counter
from difflib import SequenceMatcher

# ============================================================
# 参数设置
# ============================================================

TIME_WINDOW = 5  # 分钟
DATA_FILE = 'reddit_wsb_for_network.csv'

# 相似度阈值
SIMILARITY_THRESHOLD = 0.8  # 0.8以上视为"几乎相同"

# ============================================================
# 核心函数
# ============================================================

def text_similarity(text1, text2):
    """
    计算两个文本的相似度（0-1）
    使用SequenceMatcher（Python内置，不需要额外库）
    """
    if pd.isna(text1) or pd.isna(text2):
        return 0
    
    text1 = str(text1).lower().strip()
    text2 = str(text2).lower().strip()
    
    if text1 == '' or text2 == '':
        return 0
    
    # 使用SequenceMatcher计算相似度
    return SequenceMatcher(None, text1, text2).ratio()


def normalize_text(text):
    """
    标准化文本（用于exact match检测）
    - 转小写
    - 去除多余空格
    - 去除特殊字符（保留字母数字空格）
    """
    if pd.isna(text):
        return ''
    
    text = str(text).lower().strip()
    
    # 简单标准化：只保留字母、数字、空格
    import re
    text = re.sub(r'[^a-z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text)
    
    return text.strip()


def extract_text_duplication_features(window_data):
    """
    从时间窗口数据中提取文本重复特征
    """
    
    if len(window_data) == 0:
        return {
            'exact_duplicate_rate': 0,
            'fuzzy_duplicate_rate': 0,
            'unique_text_ratio': 100,  # 空窗口视为100%独特
            'avg_text_similarity': 0,
            'max_duplicate_count': 0,
            'duplicate_cluster_size': 0
        }
    
    texts = window_data['text'].tolist()
    total_texts = len(texts)
    
    if total_texts == 0:
        return {
            'exact_duplicate_rate': 0,
            'fuzzy_duplicate_rate': 0,
            'unique_text_ratio': 100,
            'avg_text_similarity': 0,
            'max_duplicate_count': 0,
            'duplicate_cluster_size': 0
        }
    
    # ========== 1. Exact Duplicates ==========
    # 标准化后的文本
    normalized_texts = [normalize_text(t) for t in texts]
    normalized_texts = [t for t in normalized_texts if t != '']  # 过滤空文本
    
    if len(normalized_texts) == 0:
        return {
            'exact_duplicate_rate': 0,
            'fuzzy_duplicate_rate': 0,
            'unique_text_ratio': 100,
            'avg_text_similarity': 0,
            'max_duplicate_count': 0,
            'duplicate_cluster_size': 0
        }
    
    # 计数
    text_counts = Counter(normalized_texts)
    unique_texts = len(text_counts)
    
    # Exact duplicate rate
    duplicated_texts = sum(1 for count in text_counts.values() if count > 1)
    exact_dup_rate = (1 - unique_texts / len(normalized_texts)) * 100
    
    # Unique ratio
    unique_ratio = (unique_texts / len(normalized_texts)) * 100
    
    # Max duplicate count
    max_dup_count = max(text_counts.values()) if text_counts else 0
    
    # ========== 2. Fuzzy Duplicates (相似但不完全相同) ==========
    # 为了效率，只在文本数量<100时做pairwise比较
    if len(normalized_texts) > 1 and len(normalized_texts) <= 100:
        similarities = []
        fuzzy_matches = 0
        
        for i in range(len(normalized_texts)):
            for j in range(i + 1, len(normalized_texts)):
                sim = text_similarity(normalized_texts[i], normalized_texts[j])
                similarities.append(sim)
                
                if sim >= SIMILARITY_THRESHOLD:
                    fuzzy_matches += 1
        
        avg_similarity = np.mean(similarities) if similarities else 0
        
        # Fuzzy duplicate rate: 高度相似的pairs占总pairs的比例
        total_pairs = len(similarities)
        fuzzy_dup_rate = (fuzzy_matches / total_pairs * 100) if total_pairs > 0 else 0
        
    elif len(normalized_texts) > 100:
        # 文本太多，采样计算
        sample_size = min(50, len(normalized_texts))
        sample_indices = np.random.choice(len(normalized_texts), sample_size, replace=False)
        sample_texts = [normalized_texts[i] for i in sample_indices]
        
        similarities = []
        fuzzy_matches = 0
        
        for i in range(len(sample_texts)):
            for j in range(i + 1, len(sample_texts)):
                sim = text_similarity(sample_texts[i], sample_texts[j])
                similarities.append(sim)
                
                if sim >= SIMILARITY_THRESHOLD:
                    fuzzy_matches += 1
        
        avg_similarity = np.mean(similarities) if similarities else 0
        total_pairs = len(similarities)
        fuzzy_dup_rate = (fuzzy_matches / total_pairs * 100) if total_pairs > 0 else 0
        
    else:
        # 只有1个文本
        avg_similarity = 0
        fuzzy_dup_rate = 0
    
    # ========== 3. Duplicate Cluster Size ==========
    # 最大重复簇的大小（包括原文+重复）
    duplicate_cluster_size = max(text_counts.values()) if text_counts else 0
    
    return {
        'exact_duplicate_rate': exact_dup_rate,
        'fuzzy_duplicate_rate': fuzzy_dup_rate,
        'unique_text_ratio': unique_ratio,
        'avg_text_similarity': avg_similarity,
        'max_duplicate_count': max_dup_count,
        'duplicate_cluster_size': duplicate_cluster_size
    }


# ============================================================
# 主函数
# ============================================================

def main():
    start_time = datetime.now()
    
    print("\n" + "="*70)
    print("TEXT DUPLICATION FEATURES EXTRACTION (COORDINATION DETECTION)")
    print("="*70)
    
    print("\nNote: Diagnostic score was low (5.7), but may find signals in specific windows")
    
    # ========== Step 1: 加载数据 ==========
    print("\nStep 1: Loading Reddit data...")
    
    try:
        reddit_df = pd.read_csv(DATA_FILE)
        reddit_df['timestamp'] = pd.to_datetime(reddit_df['timestamp'])
        
        print(f"  ✓ Loaded {len(reddit_df):,} items")
        print(f"  Date range: {reddit_df['timestamp'].min()} to {reddit_df['timestamp'].max()}")
        
        if 'text' not in reddit_df.columns:
            print(f"  ✗ No 'text' column found!")
            return
        
    except FileNotFoundError:
        print(f"  ✗ {DATA_FILE} not found!")
        return
    
    # ========== Step 2: 创建时间窗口 ==========
    print("\nStep 2: Creating time windows...")
    
    reddit_df['time_window'] = reddit_df['timestamp'].dt.floor(f'{TIME_WINDOW}min')
    time_windows = sorted(reddit_df['time_window'].unique())
    
    print(f"  ✓ Created {len(time_windows):,} time windows")
    
    # ========== Step 3: 逐窗口提取特征 ==========
    print("\nStep 3: Extracting text duplication features...")
    print(f"  Similarity threshold: {SIMILARITY_THRESHOLD}")
    
    features_list = []
    
    for i, window_time in enumerate(time_windows):
        if i % 100 == 0:
            print(f"  Processing window {i+1:,}/{len(time_windows):,}...", end='\r')
        
        # 获取该窗口的数据
        window_data = reddit_df[reddit_df['time_window'] == window_time].copy()
        
        # 提取特征
        window_features = extract_text_duplication_features(window_data)
        window_features['timestamp'] = window_time
        
        features_list.append(window_features)
    
    print(f"\n  ✓ Extracted features for {len(features_list):,} windows")
    
    # 转换为DataFrame
    duplication_df = pd.DataFrame(features_list)
    
    # ========== Step 4: 统计分析 ==========
    print("\nStep 4: Feature statistics...")
    
    print(f"\n  Exact duplication:")
    print(f"    Mean rate: {duplication_df['exact_duplicate_rate'].mean():.1f}%")
    print(f"    Max rate: {duplication_df['exact_duplicate_rate'].max():.1f}%")
    print(f"    Windows with >10%: {(duplication_df['exact_duplicate_rate'] > 10).sum():,}")
    print(f"    Windows with >20%: {(duplication_df['exact_duplicate_rate'] > 20).sum():,}")
    
    print(f"\n  Fuzzy duplication (similarity >{SIMILARITY_THRESHOLD}):")
    print(f"    Mean rate: {duplication_df['fuzzy_duplicate_rate'].mean():.1f}%")
    print(f"    Max rate: {duplication_df['fuzzy_duplicate_rate'].max():.1f}%")
    print(f"    Windows with >15%: {(duplication_df['fuzzy_duplicate_rate'] > 15).sum():,}")
    print(f"    Windows with >30%: {(duplication_df['fuzzy_duplicate_rate'] > 30).sum():,}")
    
    print(f"\n  Text similarity:")
    print(f"    Mean: {duplication_df['avg_text_similarity'].mean():.3f}")
    print(f"    Max: {duplication_df['avg_text_similarity'].max():.3f}")
    
    print(f"\n  Duplicate clusters:")
    print(f"    Mean max count: {duplication_df['max_duplicate_count'].mean():.1f}")
    print(f"    Max count observed: {duplication_df['max_duplicate_count'].max()}")
    print(f"    Windows with clusters >5: {(duplication_df['duplicate_cluster_size'] > 5).sum():,}")
    print(f"    Windows with clusters >10: {(duplication_df['duplicate_cluster_size'] > 10).sum():,}")
    
    # ========== Step 5: 保存 ==========
    print("\nStep 5: Saving...")
    
    duplication_df.to_parquet('text_duplication_features_5min.parquet precise', index=False)
    
    print(f"  ✓ Saved: text_duplication_features_5min.parquet precise")
    print(f"  Shape: {duplication_df.shape}")
    
    # ========== 总结 ==========
    end_time = datetime.now()
    duration = (end_time - start_time).total_seconds()
    
    print("\n" + "="*70)
    print("COMPLETE")
    print("="*70)
    
    print(f"\nTime windows: {len(duplication_df):,}")
    print(f"Feature columns: {len(duplication_df.columns)}")
    
    print(f"\nFeatures extracted:")
    print(f"  ✓ exact_duplicate_rate (copy-paste detection)")
    print(f"  ✓ fuzzy_duplicate_rate (similar text detection)")
    print(f"  ✓ unique_text_ratio (originality measure)")
    print(f"  ✓ avg_text_similarity (coordination indicator)")
    print(f"  ✓ max_duplicate_count (repetition intensity)")
    print(f"  ✓ duplicate_cluster_size (coordinated messaging)")
    
    print(f"\nCoordination signals detected:")
    high_exact = (duplication_df['exact_duplicate_rate'] > 10).sum()
    high_fuzzy = (duplication_df['fuzzy_duplicate_rate'] > 15).sum()
    large_clusters = (duplication_df['duplicate_cluster_size'] > 5).sum()
    
    print(f"  Windows with high exact duplication (>10%): {high_exact:,}")
    print(f"  Windows with high fuzzy duplication (>15%): {high_fuzzy:,}")
    print(f"  Windows with large duplicate clusters (>5): {large_clusters:,}")
    
    total_suspicious = len(duplication_df[
        (duplication_df['exact_duplicate_rate'] > 10) |
        (duplication_df['fuzzy_duplicate_rate'] > 15) |
        (duplication_df['duplicate_cluster_size'] > 5)
    ])
    
    print(f"\n  Total suspicious windows: {total_suspicious:,} ({total_suspicious/len(duplication_df)*100:.1f}%)")
    
    print(f"\nRuntime: {duration:.1f} seconds")
    
    print("\n✓ Text duplication feature extraction complete!")
    print("  (Alignment will be done in merge_all_features.py)")
    
    # ========== 解释 ==========
    print("\n" + "="*70)
    print("INTERPRETATION GUIDE")
    print("="*70)
    
    print("\nCoordination indicators:")
    print("  • High exact duplication (>10%): Copy-paste coordination")
    print("  • High fuzzy duplication (>15%): Coordinated messaging with variations")
    print("  • Large clusters (>5): Organized campaign (same message repeated)")
    print("  • High avg similarity (>0.5): Templated messages")
    
    print("\nOrganic discussion patterns:")
    print("  • Low duplication (<5%): Original content")
    print("  • High unique ratio (>90%): Diverse perspectives")
    print("  • Small clusters (<3): Natural repetition")
    
    # ========== 信号评估 ==========
    if total_suspicious > 0:
        print("\n" + "="*70)
        print("✓ SIGNAL DETECTED!")
        print("="*70)
        print(f"\nDespite low diagnostic score (5.7), found coordination signals in")
        print(f"{total_suspicious:,} windows ({total_suspicious/len(duplication_df)*100:.1f}% of total)")
        print("\nThis suggests:")
        print("  • Overall text is diverse (as diagnostic showed)")
        print("  • BUT specific windows show strong coordination")
        print("  • Text duplication features may still be valuable!")
    else:
        print("\n" + "="*70)
        print("⚠️  LIMITED SIGNAL")
        print("="*70)
        print("\nNo strong coordination signals detected.")
        print("Consistent with low diagnostic score (5.7)")
        print("\nRecommendation:")
        print("  • Keep these features (already computed)")
        print("  • Focus on Burstiness + User Overlap for main story")
        print("  • Use text duplication as supplementary evidence")


if __name__ == "__main__":
    main()


TEXT DUPLICATION FEATURES EXTRACTION (COORDINATION DETECTION)

Note: Diagnostic score was low (5.7), but may find signals in specific windows

Step 1: Loading Reddit data...
  ✓ Loaded 1,606,093 items
  Date range: 2019-07-01 04:35:02 to 2021-06-29 23:58:28

Step 2: Creating time windows...
  ✓ Created 77,267 time windows

Step 3: Extracting text duplication features...
  Similarity threshold: 0.8
  Processing window 77,201/77,267...
  ✓ Extracted features for 77,267 windows

Step 4: Feature statistics...

  Exact duplication:
    Mean rate: 0.6%
    Max rate: 68.0%
    Windows with >10%: 1,003
    Windows with >20%: 330

  Fuzzy duplication (similarity >0.8):
    Mean rate: 0.3%
    Max rate: 100.0%
    Windows with >15%: 325
    Windows with >30%: 182

  Text similarity:
    Mean: 0.139
    Max: 1.000

  Duplicate clusters:
    Mean max count: 1.3
    Max count observed: 243
    Windows with clusters >5: 1,180
    Windows with clusters >10: 377

Step 5: Saving...
  ✓ Saved: text_dup